# 01. Данные и контроль качества

Источник: выгрузка `tajikistan_soil_ml_export.gpkg` (10 складов, 169 точек XRF, 9 элементов). Блокнот не обращается ни к БД, ни к интернету. Ниже воспроизводится журнал контроля качества, переданный заказчику.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.3})
import matplotlib.pyplot as plt
import mltj
sites, pts, cov = mltj.load()
print(len(sites), 'складов,', len(pts), 'точек')

## Контуры складов

In [ ]:
g = sites.copy()
g['area_ha'] = g.area/1e4
g['valid'] = g.is_valid
n_overlap = sum(a.intersects(b) and a.intersection(b).area>0 for i,a in enumerate(g.geometry) for b in g.geometry[i+1:])
print('валидных контуров:', int(g.valid.sum()), 'из', len(g), '; пересекающихся пар:', n_overlap)
g[['name','area_ha','priority_rank']].round(2).sort_values('priority_rank')

## Положение точек относительно контура

Значительная часть точек лежит снаружи контура; расстояние до контура используется как признак модели.

In [ ]:
d = pts.assign(dist_out=pts.signed_dist_boundary_m.clip(lower=0))
print('внутри контура:', int(pts.inside.sum()), 'из', len(pts))
print('медиана расстояния до контура по всем 169 точкам (внутри = 0), м:', round(d.dist_out.median(),1))
print('медиана расстояния только у точек снаружи (%d), м:' % int((~pts.inside).sum()), round(d.loc[~d.inside,'dist_out'].median(),1))
for t in (50,100): print(f'дальше {t} м:', int((d.dist_out>t).sum()))
print('максимум, м:', round(d.dist_out.max(),1))
fig,ax=plt.subplots(figsize=(6,3.2)); ax.hist(pts.signed_dist_boundary_m,bins=30,color='#3b6ea5'); ax.axvline(0,color='k'); ax.set_xlabel('расстояние до контура, м (<0 - внутри)'); ax.set_ylabel('точек'); plt.show()

## Диапазоны значений, мг/кг

In [ ]:
t = pts[[f'{e}_mgkg' for e in mltj.ELEMENTS]].describe().T[['count','min','50%','max']]
t.index=mltj.ELEMENTS; t.columns=['значений','мин','медиана','макс']; t.round(1)

## Пропуски ковариат

Пропуски в признаках заполняются средним по выборке; доля заполненных значений указывается в model card.

In [ ]:
c = cov[['dist_to_river_m','dist_to_building_m','soil_ph','population_500m','elevation_m','slope_deg']].isna()
res = pd.DataFrame({'пустых складов из 10': c.sum()})
print('склады без pH:', list(cov.index[cov.soil_ph.isna()].map(sites.name)))
res

## Карта точек и контуров (локально, без подложки из интернета)

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
wgs=sites.to_crs(4326); p=pts.to_crs(4326)
wgs.boundary.plot(ax=ax,color='#c0392b',lw=1.2); ax.scatter(p.geometry.x,p.geometry.y,s=6,c='#222')
for r in wgs.itertuples(): ax.annotate(r.name,(r.geometry.centroid.x,r.geometry.centroid.y),fontsize=8,xytext=(3,3),textcoords='offset points')
ax.set_xlabel('долгота'); ax.set_ylabel('широта'); ax.set_title('10 складов, 169 точек'); plt.show()